<a href="https://colab.research.google.com/github/Phantom-dcode/reasoning-preserving-quantization/blob/main/quant_reasoning_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantization × reasoning: full experiment run
Runs everything the reviewers asked for, on a Colab **T4 GPU**:
1. A validated harness (chat template, `\boxed{}` extraction, 512 new tokens) on the **full GSM8K test set (1,319)**.
2. fp16, uniform NF4 and standalone INT8 baselines.
3. C1 mixed precision with an **audit** that proves the protected blocks really stay in fp16 (memory + module types).
4. C2 verification-gated fallback with a multi-line arithmetic checker (computed from the NF4 and INT8 runs).
5. C3 clamping (CoT and generic calibration), the combination, and the **H1/H2 ablations**
   (single-layer, first-k, massive-dimension exemption, constrained max-clamp with joint calibration, KL divergence).
6. Wilson intervals, exact McNemar tests and power, written to `results.json`.

**How to run:** Runtime → Change runtime type → T4 GPU. Run the install cell, then *Runtime → Run all*.
First run with `SMOKE = True` (≈10 min) to check that fp16 accuracy is near 70%; then set `SMOKE = False`.
Results are saved after every configuration, so a disconnected session resumes where it stopped.

In [ ]:
# Install pinned libraries (about 1 minute)
!pip -q install "transformers==5.17.0" "bitsandbytes>=0.45" accelerate datasets scipy
import torch; print(torch.__version__, "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.1 MB/s eta 0:00:00
2.11.0+cu130 GPU: Tesla T4


In [ ]:
import os, re, json, math, gc, time, ast, operator, random, platform, contextlib
import numpy as np
import torch
from scipy import stats as sps
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

TEST_MODE = os.environ.get("QR_TEST_MODE") == "1"   # offline pipeline test only
SMOKE = False                                          # True = 16 items per config, for a quick check

CFG = dict(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    n_main=None,            # None = all 1,319 GSM8K test items
    n_ablation=330,         # items per ablation run (80% power for a 10-point difference)
    batch_size=64,
    max_new_tokens=512,
    system_prompt="Please reason step by step, and put your final answer within \\boxed{}.",
    seed=42,
    n_calib=30,             # calibration traces (GSM8K train) for C1 probe and C3 thresholds
    n_heldout=30,           # held-out traces for KL divergence
    noise_scale=0.1,        # C1 probe: noise std = noise_scale x std of the block output
    top_frac=0.25,          # C1: fraction of blocks kept in fp16
    clamp_q=0.995,          # C3 percentile
    single_layers=[0, 1, 2, 4, 14, 27],
    first_k=[2, 4, 8, 16],
    exempt_top_m=8,         # A3: dimensions with the largest calibration max|x| exempted per layer
    constrained_k=[3, 6],   # A4: C = max(P99.5, mu + k*sigma), joint calibration
    reference_fp16_acc=73.2,
    save_dir="/content/drive/MyDrive/quant_reasoning_runs",
)
if SMOKE:
    CFG.update(n_main=16, n_ablation=8, save_dir=CFG["save_dir"] + "_smoke")
if TEST_MODE:
    CFG.update(model_id=os.environ["QR_TINY_MODEL"], n_main=10, n_ablation=6, batch_size=4, max_new_tokens=12,
               n_calib=3, n_heldout=3, single_layers=[0, 3], first_k=[2], exempt_top_m=2,
               save_dir=os.environ["QR_SAVE_DIR"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

if not TEST_MODE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
        CFG["save_dir"] = CFG["save_dir"].replace("/content/drive/MyDrive", "/content")
SAVE = CFG["save_dir"]
os.makedirs(SAVE, exist_ok=True)
RESULTS_PATH = os.path.join(SAVE, "results.json")
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}


def save_results():
    json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=1, default=float)


def env_info():
    import importlib
    info = dict(python=platform.python_version(), torch=torch.__version__, transformers=transformers.__version__,
                device=DEVICE, gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else None)
    for m in ["bitsandbytes", "accelerate", "datasets"]:
        try:
            info[m] = importlib.import_module(m).__version__
        except Exception:
            info[m] = None
    return info


RESULTS["env"] = env_info(); RESULTS["config"] = CFG; save_results()
print(json.dumps(RESULTS["env"], indent=1))

Mounted at /content/drive
{
 "python": "3.13.15",
 "torch": "2.11.0+cu130",
 "transformers": "5.17.0",
 "device": "cuda",
 "gpu": "Tesla T4",
 "bitsandbytes": "0.50.2",
 "accelerate": "1.15.0",
 "datasets": "4.8.5"
}


## Data, prompt and answer extraction

In [ ]:
def norm_num(s):
    if s is None:
        return None
    s = str(s).strip().replace(",", "").replace("$", "").replace("%", "").rstrip(".")
    try:
        return float(s)
    except ValueError:
        return None


def load_gsm8k(split):
    try:
        from datasets import load_dataset
        ds = load_dataset("openai/gsm8k", "main", split=split)
        rows, src = [dict(question=r["question"], answer=r["answer"]) for r in ds], "hf:openai/gsm8k"
    except Exception:
        import urllib.request
        url = ("https://raw.githubusercontent.com/openai/grade-school-math/master/"
               f"grade_school_math/data/{split}.jsonl")
        rows = [json.loads(l) for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]
        src = url
    for i, r in enumerate(rows):
        r["idx"] = i
        r["gold"] = norm_num(r["answer"].split("####")[-1])
    return rows, src


def load_generic(n):
    try:
        from datasets import load_dataset

        ds = load_dataset(
            "Salesforce/wikitext",
            "wikitext-2-raw-v1",
            split="train"
        )

        paras = [
            t.strip()
            for t in ds["text"]
            if len(t.strip()) > 400 and not t.strip().startswith("=")
        ]

        return paras[:n], "hf:Salesforce/wikitext"

    except Exception as e:
        if not TEST_MODE:
            raise

        print("WikiText loading failed:", e)

        return [
            "The river flows through the old town and past the market square."
        ] * n, "test-fallback"


test_rows, src_test = load_gsm8k("test")
train_rows, src_train = load_gsm8k("train")
rng = np.random.default_rng(CFG["seed"])
perm = rng.permutation(len(test_rows)).tolist()
MAIN = test_rows if CFG["n_main"] is None else [test_rows[i] for i in perm[:CFG["n_main"]]]
ABL = [test_rows[i] for i in perm[:CFG["n_ablation"]]]
tperm = rng.permutation(len(train_rows)).tolist()
CALIB = [train_rows[i] for i in tperm[:CFG["n_calib"]]]
HELDOUT = [train_rows[i] for i in tperm[CFG["n_calib"]:CFG["n_calib"] + CFG["n_heldout"]]]
GENERIC, src_generic = load_generic(CFG["n_calib"])
RESULTS["data"] = dict(test_source=src_test, generic_source=src_generic, n_main=len(MAIN), n_ablation=len(ABL),
                       ablation_idx=[r["idx"] for r in ABL], calib_idx=[r["idx"] for r in CALIB])
save_results()
print(f"main={len(MAIN)} ablation={len(ABL)} calib={len(CALIB)} heldout={len(HELDOUT)}")


def last_boxed(text):
    i = text.rfind("\\boxed{")
    if i < 0:
        return None
    j = k = i + len("\\boxed{")
    depth = 1
    while k < len(text) and depth:
        depth += {"{": 1, "}": -1}.get(text[k], 0)
        k += 1
    return text[j:k - 1] if depth == 0 else None


NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def extract_pred(text):
    b = last_boxed(text)
    src = b if b is not None else text
    src = re.sub(r"\\frac\{([^{}]*)\}\{([^{}]*)\}", r"\1/\2", src)
    nums = NUM_RE.findall(src)
    if not nums:
        return None, b is not None
    if b is not None and re.fullmatch(r"\s*-?\d+(?:\.\d+)?\s*/\s*\d+(?:\.\d+)?\s*", src):
        a, c = src.split("/")
        return float(a) / float(c), True
    return norm_num(nums[-1]), b is not None


def build_prompt(tok, question):
    msgs = [{"role": "system", "content": CFG["system_prompt"]}, {"role": "user", "content": question}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def trace_text(tok, row):
    sol = re.sub(r"<<[^>]*>>", "", row["answer"].split("####")[0]).strip()
    ans = row["answer"].split("####")[-1].strip()
    msgs = [{"role": "system", "content": CFG["system_prompt"]}, {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": f"{sol}\nThe final answer is \\boxed{{{ans}}}."}]
    return tok.apply_chat_template(msgs, tokenize=False)

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

main=1319 ablation=330 calib=30 heldout=30


## Generation, hooks and statistics

In [ ]:
def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(CFG["model_id"])
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


TOK = load_tokenizer()


def load_model(kind, skip=None):
    kw = dict(dtype=DTYPE, device_map=DEVICE)
    if kind == "nf4":
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE,
            llm_int8_skip_modules=skip)
    elif kind == "int8":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(CFG["model_id"], **kw)
    model.eval()
    return model


def free(model):
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def footprint_gb(model):
    return model.get_memory_footprint() / 1e9


def layers_of(model):
    return model.model.layers


def eos_ids(model):
    ids = model.generation_config.eos_token_id
    ids = set(ids if isinstance(ids, (list, tuple)) else [ids])
    ids.add(TOK.eos_token_id)
    ids.add(TOK.pad_token_id)
    return {i for i in ids if i is not None}


@contextlib.contextmanager
def hooks(model, fns):
    handles = [layers_of(model)[l].register_forward_hook(f) for l, f in fns.items()]
    try:
        yield
    finally:
        for h in handles:
            h.remove()


def _out(out, h):
    return (h,) + tuple(out[1:]) if isinstance(out, tuple) else h


def _hidden(out):
    return out[0] if isinstance(out, tuple) else out


def clamp_hook(C, exempt=None):
    def fn(mod, inp, out):
        h = _hidden(out)
        hc = h.clamp(-C, C)
        if exempt is not None:
            hc = torch.where(exempt.to(h.device), h, hc)
        return _out(out, hc)
    return fn


@torch.no_grad()
def generate(model, questions):
    stop = eos_ids(model)
    model.generation_config = GenerationConfig(
        do_sample=False, num_beams=1, max_new_tokens=CFG["max_new_tokens"], repetition_penalty=1.0,
        pad_token_id=TOK.pad_token_id, eos_token_id=sorted(stop))
    prompts = [build_prompt(TOK, q) for q in questions]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    bs = CFG["batch_size"]
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        enc = TOK([prompts[i] for i in idx], return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc)
        new = gen[:, enc["input_ids"].shape[1]:].tolist()
        for row, i in zip(new, idx):
            n = next((p for p, t in enumerate(row) if t in stop), len(row))
            out[i] = (TOK.decode(row[:n], skip_special_tokens=True), n)
    return out


def run_config(name, model, items, clamp_fns=None, meta=None):
    path = os.path.join(SAVE, f"{name}.jsonl")
    if os.path.exists(path):
        recs = [json.loads(l) for l in open(path)]
        if len(recs) == len(items):
            print(f"[{name}] loaded {len(recs)} saved records")
            return recs
    t0 = time.time()
    with hooks(model, clamp_fns or {}):
        gens = generate(model, [r["question"] for r in items])
    recs = []
    for r, (text, n) in zip(items, gens):
        pred, boxed = extract_pred(text)
        recs.append(dict(idx=r["idx"], gold=r["gold"], pred=pred, boxed=boxed, n_tokens=n, text=text,
                         correct=bool(pred is not None and r["gold"] is not None and abs(pred - r["gold"]) < 1e-4)))
    with open(path, "w") as f:
        for rec in recs:
            f.write(json.dumps(rec) + "\n")
    info = dict(seconds=round(time.time() - t0, 1), footprint_gb=round(footprint_gb(model), 3))
    info.update(meta or {})
    RESULTS.setdefault("run_meta", {})[name] = info
    save_results()
    acc = 100 * np.mean([r["correct"] for r in recs])
    print(f"[{name}] acc={acc:.1f}% n={len(recs)} {info}")
    return recs


def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))


def mcnemar(ref, other):
    """Exact two-sided McNemar test on items shared by both runs (matched by idx)."""
    a = {r["idx"]: r["correct"] for r in ref}
    b = {r["idx"]: r["correct"] for r in other}
    common = sorted(set(a) & set(b))
    b_ = sum(a[i] and not b[i] for i in common)
    c_ = sum(b[i] and not a[i] for i in common)
    p = 1.0 if b_ + c_ == 0 else float(min(1.0, sps.binomtest(min(b_, c_), b_ + c_, 0.5).pvalue))
    return dict(ref_only=b_, other_only=c_, p=p, n=len(common))


def summarize(recs):
    k, n = sum(r["correct"] for r in recs), len(recs)
    lo, hi = wilson(k, n)
    return dict(n=n, correct=k, acc=100 * k / n, ci=[lo, hi], cot_tokens=float(np.mean([r["n_tokens"] for r in recs])),
                boxed_rate=100 * float(np.mean([r["boxed"] for r in recs])),
                hit_max_tokens=100 * float(np.mean([r["n_tokens"] >= CFG["max_new_tokens"] for r in recs])))


def power_two_prop(p1, p2, n, alpha=0.05):
    z = sps.norm.ppf(1 - alpha / 2)
    pb = (p1 + p2) / 2
    se0 = math.sqrt(2 * pb * (1 - pb) / n)
    se1 = math.sqrt(p1 * (1 - p1) / n + p2 * (1 - p2) / n)
    d = abs(p2 - p1)
    return sps.norm.cdf((d - z * se0) / se1) + sps.norm.cdf((-d - z * se0) / se1)


RUNS = {}

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## Stage 1 — fp16: harness validation, C1 sensitivity probe

In [ ]:
import torch

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def seq_loss(model, text):
    ids = TOK(text, return_tensors="pt").input_ids.to(model.device)
    return model(input_ids=ids, labels=ids).loss.float().item()


@torch.no_grad()
def sensitivity_probe(model):

    texts = [trace_text(TOK, r) for r in CALIB]

    print(f"Calibration samples: {len(texts)}")
    print(f"Transformer layers: {len(layers_of(model))}")

    # Calculate baseline loss
    base_losses = []

    for t in tqdm(texts, desc="Baseline loss"):
        base_losses.append(seq_loss(model, t))

    base = float(np.mean(base_losses))

    scores = []

    # Probe each transformer layer
    for l in tqdm(range(len(layers_of(model))), desc="Layer sensitivity"):

        gen = torch.Generator(device=model.device).manual_seed(
            CFG["seed"]
        )

        def fn(mod, inp, out, gen=gen):
            h = _hidden(out)

            noise = torch.randn(
                h.shape,
                generator=gen,
                device=h.device,
                dtype=torch.float32
            )

            return _out(
                out,
                h + (
                    noise
                    * CFG["noise_scale"]
                    * h.float().std()
                ).to(h.dtype)
            )

        losses = []

        with hooks(model, {l: fn}):

            for t in texts:
                losses.append(seq_loss(model, t))

        scores.append(float(np.mean(losses)) - base)

    return base, scores


model = load_model("fp16")
RUNS["fp16"] = run_config("fp16", model, MAIN)
if "sensitivity" not in RESULTS:
    base, scores = sensitivity_probe(model)
    n_keep = max(1, round(CFG["top_frac"] * len(scores)))
    RESULTS["sensitivity"] = dict(base_loss=base, delta_loss=scores,
                                  protected=sorted(np.argsort(scores)[::-1][:n_keep].tolist()))
    save_results()
fp16_acc = summarize(RUNS["fp16"])["acc"]
RESULTS["harness_check"] = dict(fp16_acc=fp16_acc, reference=CFG["reference_fp16_acc"],
                                ok=bool(fp16_acc >= CFG["reference_fp16_acc"] - 10))
save_results()
print("Protected blocks for C1:", RESULTS["sensitivity"]["protected"])
if not RESULTS["harness_check"]["ok"] and not TEST_MODE:
    print("WARNING: fp16 accuracy is more than 10 points below the published 73.2%. "
          "Inspect a few outputs in fp16.jsonl before trusting the other results.")
free(model)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[fp16] loaded 1319 saved records
Protected blocks for C1: [2, 4, 5, 6, 7, 12, 13]


## Stage 2 — standalone INT8 (the C2 fallback model)

In [ ]:
model = load_model("int8")
RUNS["int8"] = run_config("int8", model, MAIN)
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[int8] acc=74.7% n=1319 {'seconds': 2322.0, 'footprint_gb': 1.777}


## Stage 3 — uniform NF4, massive-activation probe, C3 and ablations

In [ ]:
import torch

In [ ]:
@torch.no_grad()
def layer_stats(model, texts, only=None, clamp_fns=None):
    """Per-layer |x| statistics of decoder-layer outputs (residual stream), one sequence at a time."""
    L = len(layers_of(model))
    targets = range(L) if only is None else [only]
    buf = {l: [] for l in targets}
    dmax = {l: None for l in targets}

    def mk(l):
        def fn(mod, inp, out):
            a = _hidden(out).detach().abs().float()[0]
            buf[l].append(a.flatten().half().cpu())
            m = a.amax(dim=0).cpu()
            dmax[l] = m if dmax[l] is None else torch.maximum(dmax[l], m)
        return fn
    fns = dict(clamp_fns or {})
    for l in targets:
        if l in fns:   # stats are taken before this layer's own clamp is applied
            prev = fns[l]

            def both(mod, inp, out, prev=prev, rec=mk(l)):
                rec(mod, inp, out)
                return prev(mod, inp, out)
            fns[l] = both
        else:
            fns[l] = mk(l)
    with hooks(model, fns):
        for t in texts:
            model(input_ids=TOK(t, return_tensors="pt").input_ids.to(model.device))
    stats = {}
    for l in targets:
        v = torch.cat(buf[l]).float()
        k = max(1, int(math.ceil(CFG["clamp_q"] * v.numel())))
        p = float(torch.kthvalue(v, k).values)
        stats[l] = dict(p995=p, mean=float(v.mean()), std=float(v.std()), median=float(v.median()),
                        max=float(v.max()), top_dims=torch.topk(dmax[l], min(8, dmax[l].numel())).indices.tolist(),
                        top_vals=torch.topk(dmax[l], min(8, dmax[l].numel())).values.tolist(), dim_max=dmax[l])
    return stats


@torch.no_grad()
def mean_kl(model, clamp_fns):
    """Mean token KL(P_unclamped || P_clamped) on held-out traces for the same model."""
    tot, n = 0.0, 0
    for r in HELDOUT:
        ids = TOK(trace_text(TOK, r), return_tensors="pt").input_ids.to(model.device)
        lp = torch.log_softmax(model(input_ids=ids).logits.float(), -1)
        with hooks(model, clamp_fns):
            lq = torch.log_softmax(model(input_ids=ids).logits.float(), -1)
        tot += float((lp.exp() * (lp - lq)).sum(-1).sum())
        n += ids.shape[1]
    return tot / n


model = load_model("nf4")
RUNS["nf4"] = run_config("nf4", model, MAIN)
L = len(layers_of(model))
cot_texts = [trace_text(TOK, r) for r in CALIB]
st_cot = layer_stats(model, cot_texts)
st_gen = layer_stats(model, GENERIC)
RESULTS["layer_stats"] = {str(l): {k: v for k, v in s.items() if k != "dim_max"} for l, s in st_cot.items()}
RESULTS["thresholds"] = dict(cot=[st_cot[l]["p995"] for l in range(L)], generic=[st_gen[l]["p995"] for l in range(L)])
save_results()

C3_COT = {l: clamp_hook(st_cot[l]["p995"]) for l in range(L)}
C3_GEN = {l: clamp_hook(st_gen[l]["p995"]) for l in range(L)}
RUNS["c3_cot"] = run_config("c3_cot", model, MAIN, C3_COT)
RUNS["c3_gen"] = run_config("c3_gen", model, MAIN, C3_GEN)

# Ablations on the fixed subset (all use CoT thresholds)
RUNS["nf4_abl"] = [r for r in RUNS["nf4"] if r["idx"] in {a["idx"] for a in ABL}]
abl = {}
for l in CFG["single_layers"]:
    if l < L:
        abl[f"single_{l}"] = run_config(f"abl_single_{l}", model, ABL, {l: C3_COT[l]})
for k in CFG["first_k"]:
    if k <= L:
        abl[f"first_{k}"] = run_config(f"abl_first_{k}", model, ABL, {l: C3_COT[l] for l in range(k)})
exempt_fns = {}
for l in range(L):
    mask = torch.zeros_like(st_cot[l]["dim_max"], dtype=torch.bool)
    mask[torch.topk(st_cot[l]["dim_max"], CFG["exempt_top_m"]).indices] = True
    exempt_fns[l] = clamp_hook(st_cot[l]["p995"], mask)
abl["exempt_top_dims"] = run_config("abl_exempt", model, ABL, exempt_fns)

constrained = {}
for kk in CFG["constrained_k"]:
    fns, thr = {}, []
    for l in range(L):   # joint calibration: layer l calibrated with clamps on layers < l active
        s = layer_stats(model, cot_texts, only=l, clamp_fns=fns)[l]
        C = max(s["p995"], s["mean"] + kk * s["std"])
        thr.append(C)
        fns[l] = clamp_hook(C)
    constrained[kk] = fns
    RESULTS.setdefault("thresholds", {})[f"constrained_k{kk}"] = thr
    abl[f"constrained_k{kk}"] = run_config(f"abl_constrained_k{kk}", model, ABL, fns)

if "kl" not in RESULTS:
    RESULTS["kl"] = dict(c3_cot=mean_kl(model, C3_COT), c3_gen=mean_kl(model, C3_GEN),
                         exempt_top_dims=mean_kl(model, exempt_fns),
                         **{f"constrained_k{kk}": mean_kl(model, fns) for kk, fns in constrained.items()},
                         **{f"single_{l}": mean_kl(model, {l: C3_COT[l]}) for l in CFG["single_layers"] if l < L})
RESULTS["c3_samples"] = [r["text"][:600] for r in RUNS["c3_cot"][:3]]
RUNS.update({f"abl_{k}": v for k, v in abl.items()})
save_results()
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[nf4] acc=66.6% n=1319 {'seconds': 956.3, 'footprint_gb': 1.122}
[c3_cot] acc=0.5% n=1319 {'seconds': 1003.1, 'footprint_gb': 1.122}
[c3_gen] acc=0.9% n=1319 {'seconds': 1002.4, 'footprint_gb': 1.122}
[abl_single_0] acc=68.2% n=330 {'seconds': 256.1, 'footprint_gb': 1.122}
[abl_single_1] acc=36.7% n=330 {'seconds': 267.2, 'footprint_gb': 1.122}
[abl_single_2] acc=0.6% n=330 {'seconds': 267.6, 'footprint_gb': 1.122}
[abl_single_4] acc=0.9% n=330 {'seconds': 267.9, 'footprint_gb': 1.122}
[abl_single_14] acc=1.2% n=330 {'seconds': 267.1, 'footprint_gb': 1.122}
[abl_single_27] acc=62.7% n=330 {'seconds': 267.6, 'footprint_gb': 1.122}
[abl_first_2] acc=19.4% n=330 {'seconds': 267.3, 'footprint_gb': 1.122}
[abl_first_4] acc=1.8% n=330 {'seconds': 267.3, 'footprint_gb': 1.122}
[abl_first_8] acc=1.2% n=330 {'seconds': 267.5, 'footprint_gb': 1.122}
[abl_first_16] acc=1.5% n=330 {'seconds': 267.7, 'footprint_gb': 1.122}
[abl_exempt] acc=0.6% n=330 {'seconds': 299.2, 'footprint_gb': 1.122}
[abl_c

## Stage 4 — C1 with a module audit, and the combined configuration

In [ ]:
def audit_blocks(model):
    """Module class names per decoder block: proves which blocks are really quantized."""
    per_block = {}
    for name, mod in model.named_modules():
        m = re.match(r"model\.layers\.(\d+)\.", name + ".")
        if m and isinstance(mod, torch.nn.Linear):
            per_block.setdefault(int(m.group(1)), set()).add(type(mod).__name__)
    return {b: sorted(v) for b, v in sorted(per_block.items())}


def is_fp_block(types):
    return types == ["Linear"]


protected = RESULTS["sensitivity"]["protected"]
candidates = [
    ["lm_head"] + [rf"model\.layers\.{b}\." for b in protected],                    # anchored regex
    ["lm_head"] + [f"model.layers.{b}." for b in protected],                        # dotted prefix
    ["lm_head"] + [f"model.layers.{b}.{m}" for b in protected for m in
                   ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
                    "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"]],                 # explicit names
]
model, audit_ok = None, False
for skip in candidates:
    if model is not None:
        free(model)
    model = load_model("nf4", skip=skip)
    audit = audit_blocks(model)
    fp_blocks = sorted(b for b, t in audit.items() if is_fp_block(t))
    audit_ok = fp_blocks == sorted(protected)
    print("skip list", skip[:3], "... fp16 blocks:", fp_blocks, "OK" if audit_ok else "MISMATCH")
    if audit_ok:
        break
RESULTS["c1_audit"] = dict(protected=protected, fp16_blocks=fp_blocks, ok=audit_ok, skip_list=skip,
                           module_types={str(b): t for b, t in audit.items()},
                           footprint_gb=footprint_gb(model))
save_results()
assert audit_ok or TEST_MODE, "C1 could not be applied: protected blocks are still quantized. Do not report C1."
RUNS["c1"] = run_config("c1", model, MAIN)
comb_fns = {l: C3_COT[l] for l in range(L) if l not in protected}
RUNS["combined"] = run_config("combined", model, MAIN, comb_fns)
free(model)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

skip list ['lm_head', 'model\\.layers\\.2\\.', 'model\\.layers\\.4\\.'] ... fp16 blocks: [2, 4, 5, 6, 7, 12, 13] OK
[c1] acc=69.1% n=1319 {'seconds': 944.9, 'footprint_gb': 1.613}
[combined] acc=0.0% n=1319 {'seconds': 983.2, 'footprint_gb': 1.613}


## Stage 5 — C2 verifier (offline: NF4 outputs, INT8 regeneration for flagged items)

In [ ]:
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}


def safe_eval(expr):
    def ev(n):
        if isinstance(n, ast.Expression):
            return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return float(n.value)
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
            if isinstance(n.op, ast.Pow) and abs(ev(n.right)) > 10:
                raise ValueError("exponent too large")
            return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval"))


def normalize_math(s):
    s = re.sub(r"\\text\{[^{}]*\}", " ", s)
    for _ in range(3):
        s = re.sub(r"\\d?frac\{([^{}]*)\}\{([^{}]*)\}", r"((\1)/(\2))", s)
    for a, b in [("\\times", "*"), ("×", "*"), ("\\cdot", "*"), ("÷", "/"), ("\\div", "/"), ("\\left", ""),
                 ("\\right", ""), ("\\(", " "), ("\\)", " "), ("\\[", " "), ("\\]", " "), ("$", " "), ("\\$", " "),
                 ("−", "-"), ("^", "**"), ("{", "("), ("}", ")")]:
        s = s.replace(a, b)
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = re.sub(r"(?<=\d)\s*[xX]\s*(?=\d)", "*", s)
    return s


TAIL = re.compile(r"[\d\.\s\+\-\*/\(\)]+$")
HEAD = re.compile(r"^\s*[\(\s]*-?\d+(?:\.\d+)?(?:\s*(?:\*\*|[\+\-\*/])\s*\(?\s*-?\d+(?:\.\d+)?\s*\)?)*")


def check_statements(text):
    checked, bad = 0, 0
    for line in text.splitlines():
        parts = normalize_math(line).split("=")
        for lhs, rhs in zip(parts, parts[1:]):
            t, h = TAIL.search(lhs), HEAD.match(rhs)
            if not t or not h:
                continue
            le, re_ = t.group().strip(), h.group().strip()
            if not re.search(r"[\d\)]\s*(\*\*|[\+\-\*/])\s*[\(\d]", le):
                continue
            try:
                lv, rv = safe_eval(le), safe_eval(re_)
            except Exception:
                continue
            checked += 1
            dec = len(re_.split(".")[-1]) if "." in re_ and re.fullmatch(r"-?\d+\.\d+", re_) else 0
            tol = max(1e-6 * max(1.0, abs(rv)), 0.5 * 10 ** (-dec) if dec else 0)
            if abs(lv - rv) > tol:
                bad += 1
    return checked, bad


def c2_compose(base_runs, fallback_runs):
    fb = {r["idx"]: r for r in fallback_runs}
    out, n_flag, n_checked = [], 0, 0
    for r in base_runs:
        checked, bad = check_statements(r["text"])
        n_checked += checked
        if bad:
            n_flag += 1
            out.append(dict(fb[r["idx"]], escalated=True, base_correct=r["correct"]))
        else:
            out.append(dict(r, escalated=False, base_correct=r["correct"]))
    rescued = sum(r["escalated"] and r["correct"] and not r["base_correct"] for r in out)
    broken = sum(r["escalated"] and not r["correct"] and r["base_correct"] for r in out)
    return out, dict(escalated=n_flag, escalation_rate=100 * n_flag / len(out), rescued=rescued, broken=broken,
                     statements_checked=n_checked)


RUNS["c2"], RESULTS["c2"] = c2_compose(RUNS["nf4"], RUNS["int8"])
RUNS["combined_c2"], RESULTS["combined_c2"] = c2_compose(RUNS["combined"], RUNS["int8"])
save_results()
print("C2:", RESULTS["c2"])

C2: {'escalated': 72, 'escalation_rate': 5.458680818802123, 'rescued': 27, 'broken': 1, 'statements_checked': 3637}


## Stage 6 — summary tables and power

In [ ]:
table = {}
for name in ["fp16", "nf4", "int8", "c1", "c2", "c3_cot", "c3_gen", "combined", "combined_c2"]:
    s = summarize(RUNS[name])
    s["vs_nf4"] = mcnemar(RUNS["nf4"], RUNS[name]) if name != "nf4" else None
    s["vs_fp16"] = mcnemar(RUNS["fp16"], RUNS[name]) if name != "fp16" else None
    s["footprint_gb"] = RESULTS.get("run_meta", {}).get(name, {}).get("footprint_gb")
    table[name] = s
abl_table = {}
for name, recs in RUNS.items():
    if name.startswith("abl_") and name != "nf4_abl":
        s = summarize(recs)
        s["vs_nf4"] = mcnemar(RUNS["nf4_abl"], recs)
        abl_table[name] = s
abl_table["nf4_subset"] = summarize(RUNS["nf4_abl"])
n_main = len(MAIN)
RESULTS["main_table"] = table
RESULTS["ablation_table"] = abl_table
RESULTS["power"] = {f"n={n}": dict(d10=power_two_prop(.25, .35, n), d5=power_two_prop(.25, .30, n))
                    for n in sorted({40, len(ABL), n_main, 1319})}
save_results()

print(f"{'config':14s} {'acc':>6s} {'95% CI':>15s} {'p vs NF4':>9s} {'CoT tok':>8s} {'GB':>6s}")
for k, s in table.items():
    p = "—" if s["vs_nf4"] is None else f"{s['vs_nf4']['p']:.3g}"
    gb = "—" if s["footprint_gb"] is None else f"{s['footprint_gb']:.2f}"
    print(f"{k:14s} {s['acc']:6.1f} [{s['ci'][0]:5.1f},{s['ci'][1]:5.1f}] {p:>9s} {s['cot_tokens']:8.1f} {gb:>6s}")
print("\nAblations (subset n=%d):" % len(ABL))
for k, s in abl_table.items():
    p = "" if "vs_nf4" not in s else f"p={s['vs_nf4']['p']:.3g}"
    print(f"  {k:24s} acc={s['acc']:5.1f}  {p}")
print("\nKL:", {k: round(v, 4) for k, v in RESULTS["kl"].items()})
print("C1 audit ok:", RESULTS["c1_audit"]["ok"], "footprint", round(RESULTS["c1_audit"]["footprint_gb"], 3), "GB")
print("\nDone. Send me results.json from:", SAVE)

config            acc          95% CI  p vs NF4  CoT tok     GB
fp16             73.2 [ 70.8, 75.6]   1.6e-07    304.5   3.09
nf4              66.6 [ 64.0, 69.1]         —    268.7   1.12
int8             74.7 [ 72.3, 77.0]   8.1e-11    298.0   1.78
c1               69.1 [ 66.6, 71.6]    0.0329    291.6   1.61
c2               68.5 [ 66.0, 71.0]  2.16e-07    270.6      —
c3_cot            0.5 [  0.3,  1.1] 3.04e-258    509.5   1.12
c3_gen            0.9 [  0.5,  1.6] 8.83e-259    511.6   1.12
combined          0.0 [  0.0,  0.3] 9.92e-265    296.4   1.61
combined_c2       0.0 [  0.0,  0.3] 9.92e-265    296.4      —

Ablations (subset n=330):
  abl_single_0             acc= 68.2  p=0.389
  abl_single_1             acc= 36.7  p=4.14e-25
  abl_single_2             acc=  0.6  p=3.39e-68
  abl_single_4             acc=  0.9  p=6.75e-68
  abl_single_14            acc=  1.2  p=1.34e-67
  abl_single_27            acc= 62.7  p=0.00255
  abl_first_2              acc= 19.4  p=3.39e-41
  abl_first_

In [ ]:
import os

save_dir = "/content/drive/MyDrive/quant_reasoning_runs"

print("Available result files:\n")

for root, dirs, files in os.walk(save_dir):
    for file in files:
        if file.endswith((".json", ".jsonl")):
            print(os.path.join(root, file))

Available result files:

/content/drive/MyDrive/quant_reasoning_runs/fp16.jsonl
/content/drive/MyDrive/quant_reasoning_runs/results.json
/content/drive/MyDrive/quant_reasoning_runs/int8.jsonl
/content/drive/MyDrive/quant_reasoning_runs/nf4.jsonl
/content/drive/MyDrive/quant_reasoning_runs/c3_cot.jsonl
/content/drive/MyDrive/quant_reasoning_runs/c3_gen.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_0.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_1.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_2.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_4.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_14.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_single_27.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_first_2.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_first_4.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_first_8.jsonl
/content/drive/MyDrive/quant_reasoning_runs/abl_first_16.jsonl
/content/dr

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/quant_reasoning_runs/results.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

folder = "/content/drive/MyDrive/quant_reasoning_runs"

shutil.make_archive(
    "/content/full_experiment_results",
    "zip",
    folder
)

files.download("/content/full_experiment_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>